In [ ]:

!pip install -q transformers torch pandas

import torch
import pandas as pd
from transformers import pipeline

MODELS = [
    "sismetanin/rubert-ru-sentiment-rusentiment",
    "cardiffnlp/twitter-xlm-roberta-base-sentiment"
]

TEST_DATA = [
    ("POS", "Этот фильм просто великолепен!"),
    ("POS", "Мне очень понравился сервис."),
    ("POS", "Отличное качество и быстрая доставка."),
    ("NEG", "Это был ужасный опыт."),
    ("NEG", "Фильм разочаровал, зря потратил время."),
    ("NEG", "Качество отвратительное."),
    ("NEU", "Фильм вышел в 2020 году.")
]

results = []

for model_name in MODELS:
    print(f"\n Тестируется модель: {model_name}")
    try:
        classifier = pipeline(
            task="text-classification",
            model=model_name,
            tokenizer=model_name,
            truncation=True
        )

        for true_label, text in TEST_DATA:
            prediction = classifier(text)[0]

            results.append({
                "Модель": model_name,
                "Текст": text,
                "Истинная метка": true_label,
                "Предсказание": prediction["label"],
                "Уверенность": round(prediction["score"], 4)
            })

    except Exception as e:
        print("Ошибка при работе с моделью:", e)

df_results = pd.DataFrame(results)

print("\n Подробные результаты:")
display(df_results)

summary = (
    df_results
    .groupby("Модель")
    .agg(
        Средняя_уверенность=("Уверенность", "mean"),
        Количество_примеров=("Текст", "count")
    )
    .reset_index()
    .sort_values(by="Средняя_уверенность", ascending=False)
)

print("\n Сводная таблица качества моделей:")
display(summary)


🚀 Тестируется модель: sismetanin/rubert-ru-sentiment-rusentiment


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: sismetanin/rubert-ru-sentiment-rusentiment
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🚀 Тестируется модель: cardiffnlp/twitter-xlm-roberta-base-sentiment


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-xlm-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



📊 Подробные результаты:


,Модель,Текст,Истинная метка,Предсказание,Уверенность
0,sismetanin/rubert-ru-sentiment-rusentiment,Этот фильм просто великолепен!,POS,LABEL_2,0.9893
1,sismetanin/rubert-ru-sentiment-rusentiment,Мне очень понравился сервис.,POS,LABEL_2,0.9804
2,sismetanin/rubert-ru-sentiment-rusentiment,Отличное качество и быстрая доставка.,POS,LABEL_2,0.8324
3,sismetanin/rubert-ru-sentiment-rusentiment,Это был ужасный опыт.,NEG,LABEL_0,0.9606
4,sismetanin/rubert-ru-sentiment-rusentiment,"Фильм разочаровал, зря потратил время.",NEG,LABEL_0,0.9073
5,sismetanin/rubert-ru-sentiment-rusentiment,Качество отвратительное.,NEG,LABEL_0,0.9685
6,sismetanin/rubert-ru-sentiment-rusentiment,Фильм вышел в 2020 году.,NEU,LABEL_1,0.9611
7,cardiffnlp/twitter-xlm-roberta-base-sentiment,Этот фильм просто великолепен!,POS,positive,0.9300
8,cardiffnlp/twitter-xlm-roberta-base-sentiment,Мне очень понравился сервис.,POS,positive,0.8949
9,cardiffnlp/twitter-xlm-roberta-base-sentiment,Отличное качество и быстрая доставка.,POS,positive,0.8456



🏆 Сводная таблица качества моделей:


,Модель,Средняя_уверенность,Количество_примеров
1,sismetanin/rubert-ru-sentiment-rusentiment,0.942800,7
0,cardiffnlp/twitter-xlm-roberta-base-sentiment,0.890686,7


In [ ]:
import pandas as pd
from transformers import pipeline

MODELS = [
    "sismetanin/rubert-ru-sentiment-rusentiment",
    "cardiffnlp/twitter-xlm-roberta-base-sentiment"
]

TEXT = "Фильм был неплохим, но в целом ожидал большего."

configs = [
    {
        "Описание": "top_k=1, softmax",
        "top_k": 1,
        "function_to_apply": "softmax",
        "truncation": True,
        "padding": True,
        "max_length": 128
    },
    {
        "Описание": "top_k=3, softmax",
        "top_k": 3,
        "function_to_apply": "softmax",
        "truncation": True,
        "padding": True,
        "max_length": 128
    },
    {
        "Описание": "top_k=1, sigmoid",
        "top_k": 1,
        "function_to_apply": "sigmoid",
        "truncation": True,
        "padding": True,
        "max_length": 128
    },
    {
        "Описание": "top_k=3, sigmoid",
        "top_k": 3,
        "function_to_apply": "sigmoid",
        "truncation": True,
        "padding": True,
        "max_length": 128
    },
    {
        "Описание": "max_length=32",
        "top_k": 1,
        "function_to_apply": "softmax",
        "truncation": True,
        "padding": True,
        "max_length": 32
    },
    {
        "Описание": "max_length=512",
        "top_k": 1,
        "function_to_apply": "softmax",
        "truncation": True,
        "padding": True,
        "max_length": 512
    }
]

results = []

for model_name in MODELS:
    print(f"\n Загружаем модель: {model_name}")
    classifier = pipeline(
        task="text-classification",
        model=model_name,
        tokenizer=model_name
    )

    for cfg in configs:
        try:
            output = classifier(
                TEXT,
                top_k=cfg["top_k"],
                function_to_apply=cfg["function_to_apply"],
                truncation=cfg["truncation"],
                padding=cfg["padding"],
                max_length=cfg["max_length"]
            )
        except Exception as e:
            output = f"Ошибка: {e}"

        results.append({
            "Модель": model_name,
            "Описание конфигурации": cfg["Описание"],
            "top_k": cfg["top_k"],
            "function_to_apply": cfg["function_to_apply"],
            "truncation": cfg["truncation"],
            "padding": cfg["padding"],
            "max_length": cfg["max_length"],
            "Результат": str(output)
        })

df_params = pd.DataFrame(results)
print("\n📊 Результаты влияния параметров для всех моделей:")
display(df_params)


🔍 Загружаем модель: sismetanin/rubert-ru-sentiment-rusentiment


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: sismetanin/rubert-ru-sentiment-rusentiment
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🔍 Загружаем модель: cardiffnlp/twitter-xlm-roberta-base-sentiment


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-xlm-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



📊 Результаты влияния параметров для всех моделей:


,Модель,Описание конфигурации,top_k,function_to_apply,truncation,padding,max_length,Результат
0,sismetanin/rubert-ru-sentiment-rusentiment,"top_k=1, softmax",1,softmax,True,True,128,"[{'label': 'LABEL_2', 'score': 0.9348428845405..."
1,sismetanin/rubert-ru-sentiment-rusentiment,"top_k=3, softmax",3,softmax,True,True,128,"[{'label': 'LABEL_2', 'score': 0.9348428845405..."
2,sismetanin/rubert-ru-sentiment-rusentiment,"top_k=1, sigmoid",1,sigmoid,True,True,128,"[{'label': 'LABEL_2', 'score': 0.9700782299041..."
3,sismetanin/rubert-ru-sentiment-rusentiment,"top_k=3, sigmoid",3,sigmoid,True,True,128,"[{'label': 'LABEL_2', 'score': 0.9700782299041..."
4,sismetanin/rubert-ru-sentiment-rusentiment,max_length=32,1,softmax,True,True,32,"[{'label': 'LABEL_2', 'score': 0.9348428845405..."
5,sismetanin/rubert-ru-sentiment-rusentiment,max_length=512,1,softmax,True,True,512,"[{'label': 'LABEL_2', 'score': 0.9348428845405..."
6,cardiffnlp/twitter-xlm-roberta-base-sentiment,"top_k=1, softmax",1,softmax,True,True,128,"[{'label': 'positive', 'score': 0.825068891048..."
7,cardiffnlp/twitter-xlm-roberta-base-sentiment,"top_k=3, softmax",3,softmax,True,True,128,"[{'label': 'positive', 'score': 0.825068891048..."
8,cardiffnlp/twitter-xlm-roberta-base-sentiment,"top_k=1, sigmoid",1,sigmoid,True,True,128,"[{'label': 'positive', 'score': 0.801512837409..."
9,cardiffnlp/twitter-xlm-roberta-base-sentiment,"top_k=3, sigmoid",3,sigmoid,True,True,128,"[{'label': 'positive', 'score': 0.801512837409..."


In [ ]:

!pip install -q transformers torch pandas sentencepiece

import pandas as pd
from transformers import pipeline

MODELS = [
    "joeddav/xlm-roberta-large-xnli",
    "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7",
]

LABELS = ["образование", "бизнес", "политика"]

TEST_TEXTS = [
    ("образование", "Это курс о библиотеке Transformers."),
    ("образование", "Студенты изучают машинное обучение в университете."),
    ("бизнес", "Компания увеличила прибыль за счёт новых инвестиций."),
    ("бизнес", "Рынок акций отреагировал ростом котировок."),
    ("политика", "Президент подписал новый закон."),
    ("политика", "Выборы в парламент состоятся осенью.")
]

results = []

for model_name in MODELS:
    print(f"\n Модель: {model_name}")
    try:
        classifier = pipeline(
            "zero-shot-classification",
            model=model_name
        )

        for true_label, text in TEST_TEXTS:
            output = classifier(
                sequences=text,
                candidate_labels=LABELS
            )

            best_label = output["labels"][0]
            best_score = output["scores"][0]

            results.append({
                "Модель": model_name,
                "Текст": text,
                "Истинная тема": true_label,
                "Предсказанная тема": best_label,
                "Уверенность": round(best_score, 4)
            })

    except Exception as e:
        print(" Ошибка:", e)

df_results = pd.DataFrame(results)

print("\n Результаты классификации:")
display(df_results)

summary = (
    df_results
    .groupby("Модель")
    .agg(
        Средняя_уверенность=("Уверенность", "mean"),
        Количество_примеров=("Текст", "count")
    )
    .reset_index()
    .sort_values(by="Средняя_уверенность", ascending=False)
)

print("\n Качество моделей:")
display(summary)


🚀 Модель: joeddav/xlm-roberta-large-xnli


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: joeddav/xlm-roberta-large-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🚀 Модель: MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



📊 Результаты классификации:


,Модель,Текст,Истинная тема,Предсказанная тема,Уверенность
0,joeddav/xlm-roberta-large-xnli,Это курс о библиотеке Transformers.,образование,образование,0.9684
1,joeddav/xlm-roberta-large-xnli,Студенты изучают машинное обучение в университ...,образование,образование,0.9554
2,joeddav/xlm-roberta-large-xnli,Компания увеличила прибыль за счёт новых инвес...,бизнес,бизнес,0.9628
3,joeddav/xlm-roberta-large-xnli,Рынок акций отреагировал ростом котировок.,бизнес,бизнес,0.8766
4,joeddav/xlm-roberta-large-xnli,Президент подписал новый закон.,политика,политика,0.7386
5,joeddav/xlm-roberta-large-xnli,Выборы в парламент состоятся осенью.,политика,политика,0.8719
6,MoritzLaurer/mDeBERTa-v3-base-xnli-multilingua...,Это курс о библиотеке Transformers.,образование,образование,0.9194
7,MoritzLaurer/mDeBERTa-v3-base-xnli-multilingua...,Студенты изучают машинное обучение в университ...,образование,образование,0.9470
8,MoritzLaurer/mDeBERTa-v3-base-xnli-multilingua...,Компания увеличила прибыль за счёт новых инвес...,бизнес,бизнес,0.9877
9,MoritzLaurer/mDeBERTa-v3-base-xnli-multilingua...,Рынок акций отреагировал ростом котировок.,бизнес,бизнес,0.6330



🏆 Качество моделей:


,Модель,Средняя_уверенность,Количество_примеров
1,joeddav/xlm-roberta-large-xnli,0.895617,6
0,MoritzLaurer/mDeBERTa-v3-base-xnli-multilingua...,0.855350,6


In [ ]:
import pandas as pd
from transformers import pipeline

MODELS = [
    "joeddav/xlm-roberta-large-xnli",
    "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"
]

TEXT = "Это курс о библиотеке Transformers."
LABELS = ["образование", "бизнес", "политика"]

configs = [
    {
        "Описание": "Базовая конфигурация",
        "candidate_labels": LABELS,
        "hypothesis_template": "Этот текст относится к теме {}.",
        "multi_label": False,
        "max_length": 128
    },
    {
        "Описание": "Другой шаблон гипотезы",
        "candidate_labels": LABELS,
        "hypothesis_template": "Данная тема текста — {}.",
        "multi_label": False,
        "max_length": 128
    },
    {
        "Описание": "multi_label=True",
        "candidate_labels": LABELS,
        "hypothesis_template": "Этот текст относится к теме {}.",
        "multi_label": True,
        "max_length": 128
    },
    {
        "Описание": "max_length=32",
        "candidate_labels": LABELS,
        "hypothesis_template": "Этот текст относится к теме {}.",
        "multi_label": False,
        "max_length": 32
    },
    {
        "Описание": "max_length=512",
        "candidate_labels": LABELS,
        "hypothesis_template": "Этот текст относится к теме {}.",
        "multi_label": False,
        "max_length": 512
    }
]

param_results = []

for model_name in MODELS:
    print(f"\n Загружаем модель: {model_name}")
    classifier = pipeline(
        "zero-shot-classification",
        model=model_name
    )

    for cfg in configs:
        try:
            output = classifier(
                sequences=TEXT,
                candidate_labels=cfg["candidate_labels"],
                hypothesis_template=cfg["hypothesis_template"],
                multi_label=cfg["multi_label"],
                truncation=True,
                padding=True,
                max_length=cfg["max_length"]
            )
        except Exception as e:
            output = {"labels": ["ошибка"], "scores": [0.0], "sequence": TEXT}
            print(f"Ошибка для {model_name} с конфигом {cfg['Описание']}: {e}")

        param_results.append({
            "Модель": model_name,
            "Описание конфигурации": cfg["Описание"],
            "Предсказанная тема": output["labels"][0],
            "Уверенность": round(output["scores"][0], 4),
            "Полный результат": str(output)
        })

df_params = pd.DataFrame(param_results)

print("\n Влияние параметров на zero-shot классификацию (две модели):")
display(df_params)


🔍 Загружаем модель: joeddav/xlm-roberta-large-xnli


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: joeddav/xlm-roberta-large-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🔍 Загружаем модель: MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🔬 Влияние параметров на zero-shot классификацию (две модели):


,Модель,Описание конфигурации,Предсказанная тема,Уверенность,Полный результат
0,joeddav/xlm-roberta-large-xnli,Базовая конфигурация,образование,0.5048,{'sequence': 'Это курс о библиотеке Transforme...
1,joeddav/xlm-roberta-large-xnli,Другой шаблон гипотезы,образование,0.6534,{'sequence': 'Это курс о библиотеке Transforme...
2,joeddav/xlm-roberta-large-xnli,multi_label=True,бизнес,0.7074,{'sequence': 'Это курс о библиотеке Transforme...
3,joeddav/xlm-roberta-large-xnli,max_length=32,образование,0.5048,{'sequence': 'Это курс о библиотеке Transforme...
4,joeddav/xlm-roberta-large-xnli,max_length=512,образование,0.5048,{'sequence': 'Это курс о библиотеке Transforme...
5,MoritzLaurer/mDeBERTa-v3-base-xnli-multilingua...,Базовая конфигурация,образование,0.7005,{'sequence': 'Это курс о библиотеке Transforme...
6,MoritzLaurer/mDeBERTa-v3-base-xnli-multilingua...,Другой шаблон гипотезы,образование,0.9736,{'sequence': 'Это курс о библиотеке Transforme...
7,MoritzLaurer/mDeBERTa-v3-base-xnli-multilingua...,multi_label=True,образование,0.0653,{'sequence': 'Это курс о библиотеке Transforme...
8,MoritzLaurer/mDeBERTa-v3-base-xnli-multilingua...,max_length=32,образование,0.7005,{'sequence': 'Это курс о библиотеке Transforme...
9,MoritzLaurer/mDeBERTa-v3-base-xnli-multilingua...,max_length=512,образование,0.7005,{'sequence': 'Это курс о библиотеке Transforme...


In [ ]:

from transformers import pipeline
import pandas as pd

model_names = [
    "ai-forever/rugpt3small_based_on_gpt2",
    "ai-forever/rugpt3medium_based_on_gpt2"
]

prompts = [
    "Однажды в студёную зимнюю пору",
    "Искусственный интеллект — это",
    "Вчера я встретил старого друга, и мы"
]

results_5 = []

for model_name in model_names:
    print(f"Загружаем модель: {model_name}")
    generator = pipeline("text-generation", model=model_name, tokenizer=model_name)
    for prompt in prompts:
        output = generator(prompt, max_new_tokens=50, do_sample=True, temperature=0.8)[0]['generated_text']
        results_5.append({
            "Модель": model_name,
            "Промпт": prompt,
            "Сгенерированный текст": output
        })

df5 = pd.DataFrame(results_5)
print("Результаты задания 5:")
print(df5.to_string())


Загружаем модель: ai-forever/rugpt3small_based_on_gpt2


config.json:   0%|          | 0.00/720 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/551M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/551M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/574 [00:00<?, ?B/s]

Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Результаты задания 5:
                                  Модель                                Промпт                                                                                                                                                                                                                                                                        Сгенерированный текст
0   ai-forever/rugpt3small_based_on_gpt2        Однажды в студёную зимнюю пору                                       Однажды в студёную зимнюю пору, когда в окрестностях города, в окрестностях домов, расположенных в парках, на деревьях, на деревьях и кустарниках, а также в лесах, как в городах, так и в сёлах, где можно встретить диких животных, в лесных озерах,
1   ai-forever/rugpt3small_based_on_gpt2         Искусственный интеллект — это  Искусственный интеллект — это, прежде всего, умение работать в команде, которая не имеет права на ошибку.  Это должен быть не просто большой человек, а человек, который с

In [ ]:
import pandas as pd
import torch
from transformers import pipeline

MODELS = [
    "ai-forever/rugpt3small_based_on_gpt2",
    "ai-forever/rugpt3medium_based_on_gpt2"
]

TEST_PROMPT = "Расскажи о преимуществах искусственного интеллекта в"

configs = [
    {
        "Описание": "Базовая (greedy)",
        "max_new_tokens": 50,
        "do_sample": False,
        "temperature": None,
        "top_p": None,
        "top_k": None,
        "repetition_penalty": 1.0
    },
    {
        "Описание": "Temperature=0.7",
        "max_new_tokens": 50,
        "do_sample": True,
        "temperature": 0.7,
        "top_p": 0.9,
        "top_k": None,
        "repetition_penalty": 1.0
    },
    {
        "Описание": "Temperature=1.5 (более случайно)",
        "max_new_tokens": 50,
        "do_sample": True,
        "temperature": 1.5,
        "top_p": 0.9,
        "top_k": None,
        "repetition_penalty": 1.0
    },
    {
        "Описание": "Temperature=0.3 (более детерминировано)",
        "max_new_tokens": 50,
        "do_sample": True,
        "temperature": 0.3,
        "top_p": 0.9,
        "top_k": None,
        "repetition_penalty": 1.0
    },
    {
        "Описание": "top_p=0.8 (nucleus sampling)",
        "max_new_tokens": 50,
        "do_sample": True,
        "temperature": 0.7,
        "top_p": 0.8,
        "top_k": None,
        "repetition_penalty": 1.0
    },
    {
        "Описание": "top_k=40",
        "max_new_tokens": 50,
        "do_sample": True,
        "temperature": 0.7,
        "top_p": None,
        "top_k": 40,
        "repetition_penalty": 1.0
    },
    {
        "Описание": "repetition_penalty=1.5 (штраф за повторы)",
        "max_new_tokens": 50,
        "do_sample": True,
        "temperature": 0.7,
        "top_p": 0.9,
        "top_k": None,
        "repetition_penalty": 1.5
    },
    {
        "Описание": "max_new_tokens=20 (короткий ответ)",
        "max_new_tokens": 20,
        "do_sample": True,
        "temperature": 0.7,
        "top_p": 0.9,
        "top_k": None,
        "repetition_penalty": 1.0
    },
    {
        "Описание": "max_new_tokens=100 (длинный ответ)",
        "max_new_tokens": 100,
        "do_sample": True,
        "temperature": 0.7,
        "top_p": 0.9,
        "top_k": None,
        "repetition_penalty": 1.0
    },
    {
        "Описание": "num_beams=4 (beam search)",
        "max_new_tokens": 50,
        "do_sample": False,
        "temperature": None,
        "top_p": None,
        "top_k": None,
        "repetition_penalty": 1.0,
        "num_beams": 4
    }
]


def analyze_model(model_name):
    print(f"\n{'='*60}")
    print(f" АНАЛИЗ МОДЕЛИ: {model_name}")
    print(f"{'='*60}")


    generator = pipeline(
        "text-generation",
        model=model_name,
        device_map="auto" if torch.cuda.is_available() else None,
        trust_remote_code=True
    )

    if generator.tokenizer.pad_token_id is None:
        generator.tokenizer.pad_token_id = generator.tokenizer.eos_token_id

    results = []

    for cfg in configs:
        print(f"\n Тестируем: {cfg['Описание']}")

        gen_params = {
            "max_new_tokens": cfg["max_new_tokens"],
            "pad_token_id": generator.tokenizer.eos_token_id
        }

        if cfg.get("do_sample") is not None:
            gen_params["do_sample"] = cfg["do_sample"]
        if cfg.get("temperature") is not None:
            gen_params["temperature"] = cfg["temperature"]
        if cfg.get("top_p") is not None:
            gen_params["top_p"] = cfg["top_p"]
        if cfg.get("top_k") is not None:
            gen_params["top_k"] = cfg["top_k"]
        if cfg.get("repetition_penalty") is not None:
            gen_params["repetition_penalty"] = cfg["repetition_penalty"]
        if cfg.get("num_beams") is not None:
            gen_params["num_beams"] = cfg["num_beams"]

        try:
            output = generator(TEST_PROMPT, **gen_params)
            generated = output[0]['generated_text']

            generated_part = generated[len(TEST_PROMPT):]
            text_length = len(generated_part)

            words = generated_part.split()
            total_words = len(words)
            unique_words = len(set(word.lower() for word in words))
            lexical_diversity = unique_words / total_words if total_words > 0 else 0

            results.append({
                "Модель": model_name,
                "Конфигурация": cfg["Описание"],
                "Сгенерированный текст (начало)": generated_part[:150] + "..." if len(generated_part) > 150 else generated_part,
                "Длина_текста (символы)": text_length,
                "Лексическое_разнообразие": round(lexical_diversity, 3),
                "max_new_tokens": cfg["max_new_tokens"],
                "temperature": cfg["temperature"] if cfg.get("temperature") is not None else "N/A",
                "top_p": cfg["top_p"] if cfg.get("top_p") is not None else "N/A",
                "top_k": cfg["top_k"] if cfg.get("top_k") is not None else "N/A",
                "repetition_penalty": cfg["repetition_penalty"],
                "num_beams": cfg.get("num_beams", 1)
            })

            print(f"✓ Успешно! Длина (симв): {text_length}, Лекс. разнообразие: {lexical_diversity:.3f}")

        except Exception as e:
            print(f"✗ Ошибка: {e}")
            results.append({
                "Модель": model_name,
                "Конфигурация": cfg["Описание"],
                "Сгенерированный текст (начало)": f"ERROR: {e}",
                "Длина_текста (символы)": 0,
                "Лексическое_разнообразие": 0,
                "max_new_tokens": cfg["max_new_tokens"],
                "temperature": cfg["temperature"] if cfg.get("temperature") is not None else "N/A",
                "top_p": cfg["top_p"] if cfg.get("top_p") is not None else "N/A",
                "top_k": cfg["top_k"] if cfg.get("top_k") is not None else "N/A",
                "repetition_penalty": cfg["repetition_penalty"],
                "num_beams": cfg.get("num_beams", 1)
            })

    return pd.DataFrame(results)

all_results = pd.DataFrame()
for model in MODELS:
    df_model = analyze_model(model)
    all_results = pd.concat([all_results, df_model], ignore_index=True)

all_results.to_csv("models_parameters_comparison.csv", index=False, encoding='utf-8-sig')

print("\n" + "="*80)
print("📊 СРАВНЕНИЕ МОДЕЛЕЙ ПО ВСЕМ КОНФИГУРАЦИЯМ")
print("="*80)


pivot_length = all_results.pivot(index="Конфигурация", columns="Модель", values="Длина_текста (символы)")
pivot_diversity = all_results.pivot(index="Конфигурация", columns="Модель", values="Лексическое_разнообразие")

print("\n🔹 Длина сгенерированного текста (в символах):")
display(pivot_length)

print("\n🔹 Лексическое разнообразие:")
display(pivot_diversity)



🔬 АНАЛИЗ МОДЕЛИ: ai-forever/rugpt3small_based_on_gpt2


Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: ai-forever/rugpt3small_based_on_gpt2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 
transformer.h.{0...11}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📝 Тестируем: Базовая (greedy)


Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✓ Успешно! Длина (симв): 105, Лекс. разнообразие: 1.000

📝 Тестируем: num_beams=4 (beam search)
✓ Успешно! Длина (симв): 57, Лекс. разнообразие: 1.000

📊 СРАВНЕНИЕ МОДЕЛЕЙ ПО ВСЕМ КОНФИГУРАЦИЯМ

🔹 Длина сгенерированного текста (в символах):


Модель,ai-forever/rugpt3medium_based_on_gpt2,ai-forever/rugpt3small_based_on_gpt2
Конфигурация,,
Temperature=0.3 (более детерминировано),63,179
Temperature=0.7,147,68
Temperature=1.5 (более случайно),232,246
max_new_tokens=100 (длинный ответ),105,415
max_new_tokens=20 (короткий ответ),33,102
num_beams=4 (beam search),57,56
repetition_penalty=1.5 (штраф за повторы),220,354
top_k=40,264,229
top_p=0.8 (nucleus sampling),278,242



🔹 Лексическое разнообразие:


Модель,ai-forever/rugpt3medium_based_on_gpt2,ai-forever/rugpt3small_based_on_gpt2
Конфигурация,,
Temperature=0.3 (более детерминировано),1.000,0.611
Temperature=0.7,0.733,1.000
Temperature=1.5 (более случайно),0.902,0.889
max_new_tokens=100 (длинный ответ),1.000,0.672
max_new_tokens=20 (короткий ответ),1.000,1.000
num_beams=4 (beam search),1.000,1.000
repetition_penalty=1.5 (штраф за повторы),0.939,0.977
top_k=40,0.971,0.871
top_p=0.8 (nucleus sampling),0.935,0.718


In [ ]:

!pip install -q transformers torch pandas

import pandas as pd
import torch
import time
from transformers import pipeline

MODELS = [
    "deepvk/RuModernBERT-small",
    "DeepPavlov/rubert-base-cased",
]

TEST_SENTENCES = [
    "Москва — это [MASK] России.",
    "Кошка поймала [MASK].",
    "Сегодня отличная [MASK] для прогулки.",
    "Программист пишет [MASK] на Python.",
    "В холодильнике есть [MASK] и молоко."
]

print("="*80)
print(" ТЕСТИРОВАНИЕ FILL-MASK МОДЕЛЕЙ")
print("="*80)

all_results = []
loading_times = []

for model_name in MODELS:
    print(f"\n Загрузка модели: {model_name}")
    start_load = time.time()

    try:
        unmasker = pipeline(
            "fill-mask",
            model=model_name,
            device=0 if torch.cuda.is_available() else -1,
            top_k=5
        )

        load_time = time.time() - start_load
        loading_times.append({
            "Модель": model_name,
            "Время загрузки (сек)": round(load_time, 2)
        })

        print(f" Загружена за {load_time:.2f} сек")
        for sentence in TEST_SENTENCES:
            result = unmasker(sentence)

            best_prediction = result[0] if isinstance(result, list) else result

            all_results.append({
                "Модель": model_name,
                "Предложение": sentence,
                "Заполнение": best_prediction.get('token_str', 'N/A'),
                "Уверенность": round(best_prediction.get('score', 0), 4),
                "Все варианты": ", ".join([r.get('token_str', '') for r in result[:3]]) if isinstance(result, list) else "N/A"
            })

        del unmasker
        torch.cuda.empty_cache()

    except Exception as e:
        print(f" Ошибка: {e}")
        loading_times.append({
            "Модель": model_name,
            "Время загрузки (сек)": None
        })

df_loading = pd.DataFrame(loading_times)
print("\n" + "="*80)
print("⏱ ВРЕМЯ ЗАГРУЗКИ МОДЕЛЕЙ")
print("="*80)
display(df_loading.sort_values(by="Время загрузки (сек)", na_position='last'))

df_results = pd.DataFrame(all_results)
print("\n" + "="*80)
print(" РЕЗУЛЬТАТЫ ЗАПОЛНЕНИЯ МАСКИ")
print("="*80)
display(df_results.head(15))

summary = (
    df_results
    .groupby("Модель")
    .agg(
        Средняя_уверенность=("Уверенность", "mean"),
        Количество_успешных=("Уверенность", lambda x: (x > 0).sum()),
        Пример_заполнения=("Заполнение", "first")
    )
    .reset_index()
    .merge(df_loading, on="Модель")
    .sort_values(by="Средняя_уверенность", ascending=False)
)

print("\n" + "="*80)
print(" СВОДНАЯ ТАБЛИЦА КАЧЕСТВА МОДЕЛЕЙ")
print("="*80)
display(summary)

df_results.to_csv("fillmask_models_comparison.csv", index=False, encoding='utf-8-sig')
summary.to_csv("fillmask_models_summary.csv", index=False, encoding='utf-8-sig')

print("\n Результаты сохранены в CSV файлы")

 ТЕСТИРОВАНИЕ FILL-MASK МОДЕЛЕЙ

🚀 Загрузка модели: deepvk/RuModernBERT-small


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/138M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/77 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...


📝 РЕЗУЛЬТАТЫ ЗАПОЛНЕНИЯ МАСКИ


,Модель,Предложение,Заполнение,Уверенность,Все варианты
0,deepvk/RuModernBERT-small,Москва — это [MASK] России.,столица,0.8066,"столица, город, в"
1,deepvk/RuModernBERT-small,Кошка поймала [MASK].,собаку,0.0978,"собаку, меня, кота"
2,deepvk/RuModernBERT-small,Сегодня отличная [MASK] для прогулки.,погода,0.8689,"погода, идея, компания"
3,deepvk/RuModernBERT-small,Программист пишет [MASK] на Python.,код,0.5045,"код, текст, тесты"
4,deepvk/RuModernBERT-small,В холодильнике есть [MASK] и молоко.,хлеб,0.2261,"хлеб, яйца, творог"
5,DeepPavlov/rubert-base-cased,Москва — это [MASK] России.,столица,0.4760,"столица, символ, гордость"
6,DeepPavlov/rubert-base-cased,Кошка поймала [MASK].,животное,0.0716,"животное, собаку, змею"
7,DeepPavlov/rubert-base-cased,Сегодня отличная [MASK] для прогулки.,возможность,0.5071,"возможность, площадка, дорога"
8,DeepPavlov/rubert-base-cased,Программист пишет [MASK] на Python.,программы,0.1958,"программы, тексты, статьи"
9,DeepPavlov/rubert-base-cased,В холодильнике есть [MASK] и молоко.,мясо,0.1401,"мясо, чай, вода"



🏆 СВОДНАЯ ТАБЛИЦА КАЧЕСТВА МОДЕЛЕЙ


,Модель,Средняя_уверенность,Количество_успешных,Пример_заполнения,Время загрузки (сек)
1,deepvk/RuModernBERT-small,0.50078,5,столица,15.2
0,DeepPavlov/rubert-base-cased,0.27812,5,столица,15.3



✅ Результаты сохранены в CSV файлы


In [ ]:
import pandas as pd
import torch
from transformers import pipeline

MODELS = [
    "deepvk/RuModernBERT-small",
    "DeepPavlov/rubert-base-cased"
]

configs = [
    {
        "Описание": "Базовая (top_k=5)",
        "top_k": 5,
        "targets": None,
        "batch_size": None
    },
    {
        "Описание": "top_k=1 (только лучший)",
        "top_k": 1,
        "targets": None,
        "batch_size": None
    },
    {
        "Описание": "top_k=10 (больше вариантов)",
        "top_k": 10,
        "targets": None,
        "batch_size": None
    },
    {
        "Описание": "Конкретные targets",
        "top_k": 5,
        "targets": ["собаку", "кошку", "мышь"],
        "batch_size": None
    },
    {
        "Описание": "batch_size=4",
        "top_k": 5,
        "targets": None,
        "batch_size": 4
    },
    {
        "Описание": "batch_size=1",
        "top_k": 5,
        "targets": None,
        "batch_size": 1
    }
]

TEST_SENTENCES = [
    "Ребенок мыл [MASK].",
    "Я люблю есть [MASK] на завтрак.",
    "В парке растёт красивый [MASK]."
]

all_results = []

for model_name in MODELS:
    print(f"\n{'='*60}")
    print(f" АНАЛИЗ МОДЕЛИ: {model_name}")
    print('='*60)

    for cfg in configs:
        print(f"\n Тестируем: {cfg['Описание']}")

        try:
            unmasker = pipeline(
                "fill-mask",
                model=model_name,
                top_k=cfg["top_k"],
                targets=cfg["targets"],
                batch_size=cfg["batch_size"] if cfg["batch_size"] else None,
                device=0 if torch.cuda.is_available() else -1
            )

            model_results = []
            for sentence in TEST_SENTENCES:
                result = unmasker(sentence)
                best = result[0] if isinstance(result, list) else result
                model_results.append({
                    "Предложение": sentence,
                    "Заполнение": best.get('token_str', 'N/A'),
                    "Уверенность": round(best.get('score', 0), 4),
                    "Количество вариантов": len(result) if isinstance(result, list) else 1
                })

            avg_confidence = sum(r["Уверенность"] for r in model_results) / len(model_results)
            avg_variants = sum(r["Количество вариантов"] for r in model_results) / len(model_results)

            all_results.append({
                "Модель": model_name,
                "Конфигурация": cfg["Описание"],
                "top_k": cfg["top_k"],
                "targets": "Да" if cfg["targets"] else "Нет",
                "batch_size": cfg["batch_size"] if cfg["batch_size"] else "auto",
                "Средняя уверенность": round(avg_confidence, 4),
                "Среднее кол-во вариантов": round(avg_variants, 1),
                "Пример 1": model_results[0]["Заполнение"],
                "Пример 2": model_results[1]["Заполнение"],
                "Пример 3": model_results[2]["Заполнение"]
            })

            print(f"✓ Успешно! Средняя уверенность: {avg_confidence:.4f}")

            del unmasker
            torch.cuda.empty_cache()

        except Exception as e:
            print(f"✗ Ошибка: {e}")
            all_results.append({
                "Модель": model_name,
                "Конфигурация": cfg["Описание"],
                "top_k": cfg["top_k"],
                "targets": "Да" if cfg["targets"] else "Нет",
                "batch_size": cfg["batch_size"] if cfg["batch_size"] else "auto",
                "Средняя уверенность": 0,
                "Среднее кол-во вариантов": 0,
                "Пример 1": f"ERROR: {e}",
                "Пример 2": "N/A",
                "Пример 3": "N/A"
            })

df_all = pd.DataFrame(all_results)

print("\n" + "="*80)
print(" СВОДНАЯ ТАБЛИЦА ПО ВСЕМ МОДЕЛЯМ И ПАРАМЕТРАМ")
print("="*80)
display(df_all)

print("\n" + "="*80)
print(" СРАВНЕНИЕ МОДЕЛЕЙ ПО УВЕРЕННОСТИ")
print("="*80)

pivot_conf = df_all.pivot_table(
    index="Конфигурация",
    columns="Модель",
    values="Средняя уверенность",
    aggfunc="first"
)
display(pivot_conf)

print("\n" + "="*80)
print(" СРАВНЕНИЕ МОДЕЛЕЙ ПО КОЛИЧЕСТВУ ВАРИАНТОВ")
print("="*80)

pivot_variants = df_all.pivot_table(
    index="Конфигурация",
    columns="Модель",
    values="Среднее кол-во вариантов",
    aggfunc="first"
)
display(pivot_variants)

print("\n" + "="*80)
print(" ЛУЧШИЕ КОНФИГУРАЦИИ ДЛЯ КАЖДОЙ МОДЕЛИ")
print("="*80)

for model in MODELS:
    df_model = df_all[df_all["Модель"] == model]
    best_conf = df_model.loc[df_model["Средняя уверенность"].idxmax()]
    print(f"\n {model}")
    print(f"   Лучшая конфигурация: {best_conf['Конфигурация']}")
    print(f"   Уверенность: {best_conf['Средняя уверенность']:.4f}")
    print(f"   Вариантов: {best_conf['Среднее кол-во вариантов']:.1f}")



🔬 АНАЛИЗ МОДЕЛИ: deepvk/RuModernBERT-small

📝 Тестируем: Базовая (top_k=5)


Loading weights:   0%|          | 0/77 [00:00<?, ?it/s]

✓ Успешно! Средняя уверенность: 0.1908

📝 Тестируем: top_k=1 (только лучший)


Loading weights:   0%|          | 0/77 [00:00<?, ?it/s]

✓ Успешно! Средняя уверенность: 0.1908

📝 Тестируем: top_k=10 (больше вариантов)


...


🔍 СРАВНЕНИЕ МОДЕЛЕЙ ПО УВЕРЕННОСТИ


Модель,DeepPavlov/rubert-base-cased,deepvk/RuModernBERT-small
Конфигурация,,
batch_size=1,0.2392,0.1908
batch_size=4,0.2392,0.1908
top_k=1 (только лучший),0.2392,0.1908
top_k=10 (больше вариантов),0.2392,0.1908
Базовая (top_k=5),0.2392,0.1908
Конкретные targets,0.0007,0.0000



🔍 СРАВНЕНИЕ МОДЕЛЕЙ ПО КОЛИЧЕСТВУ ВАРИАНТОВ


Модель,DeepPavlov/rubert-base-cased,deepvk/RuModernBERT-small
Конфигурация,,
batch_size=1,5.0,5.0
batch_size=4,5.0,5.0
top_k=1 (только лучший),1.0,1.0
top_k=10 (больше вариантов),10.0,10.0
Базовая (top_k=5),5.0,5.0
Конкретные targets,3.0,3.0



🏆 ЛУЧШИЕ КОНФИГУРАЦИИ ДЛЯ КАЖДОЙ МОДЕЛИ

✅ deepvk/RuModernBERT-small
   Лучшая конфигурация: Базовая (top_k=5)
   Уверенность: 0.1908
   Вариантов: 5.0

✅ DeepPavlov/rubert-base-cased
   Лучшая конфигурация: Базовая (top_k=5)
   Уверенность: 0.2392
   Вариантов: 5.0
